In [35]:
from pathlib import Path
import pandas as pd
import numpy as np

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

print("Shape:", df.shape)
print()
print(df["split"].value_counts())

Shape: (12515, 24)

split
train         8789
test          1866
validation    1860
Name: count, dtype: int64


In [36]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

spo2_hr_features = spo2_features + [
    "heart_rate",
    "heart_rate_lag_1",
    "heart_rate_rolling_mean_5"
]

target = "future_spo2_5min"

In [37]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

train_ready = train_df.dropna(
    subset=[target]
).copy()

val_ready = val_df.dropna(
    subset=[
        "spo2",
        target
    ]
).copy()

X_train = train_ready[spo2_hr_features]
y_train = train_ready[target]

X_val = val_ready[spo2_hr_features]
y_val = val_ready[target]

groups = train_ready["caseid"]

print("Training rows:", len(X_train))
print("Training patients:", groups.nunique())
print("Validation rows:", len(X_val))

Training rows: 8310
Training patients: 35
Validation rows: 1748


In [38]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import (
    GroupKFold,
    GridSearchCV
)

In [39]:
pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

In [40]:
param_grid = {
    "model__n_estimators": [
        100,
        200
    ],

    "model__learning_rate": [
        0.03,
        0.05
    ],

    "model__max_depth": [
        2,
        3
    ],

    "model__min_samples_leaf": [
        3
    ]
}

In [41]:
group_cv = GroupKFold(
    n_splits=5
)

In [42]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=group_cv,
    n_jobs=-1,
    verbose=1
)

In [43]:
grid_search.fit(
    X_train,
    y_train,
    groups=groups
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.03, 0.05], 'model__max_depth': [2, 3], 'model__min_samples_leaf': [3], 'model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",GroupKFold(n_...shuffle=False)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know mor

In [44]:
print("Best parameters:")
print(grid_search.best_params_)

print()

print(
    "Best CV RMSE:",
    -grid_search.best_score_
)

Best parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__min_samples_leaf': 3, 'model__n_estimators': 200}

Best CV RMSE: 1.8818898299247255


In [45]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [46]:
best_model = grid_search.best_estimator_

val_predictions = best_model.predict(
    X_val
)

tuned_mae = mean_absolute_error(
    y_val,
    val_predictions
)

tuned_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        val_predictions
    )
)

tuned_r2 = r2_score(
    y_val,
    val_predictions
)

print("=== TUNED GRADIENT BOOSTING + HR ===")
print()

print(f"MAE:  {tuned_mae:.6f}")
print(f"RMSE: {tuned_rmse:.6f}")
print(f"R²:   {tuned_r2:.6f}")

=== TUNED GRADIENT BOOSTING + HR ===

MAE:  0.452540
RMSE: 1.336950
R²:   0.151922


In [47]:
print("Best parameters:")
print(grid_search.best_params_)

print()

print(
    "Best cross-validation RMSE:",
    -grid_search.best_score_
)

Best parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__min_samples_leaf': 3, 'model__n_estimators': 200}

Best cross-validation RMSE: 1.8818898299247255


In [48]:
best_model = grid_search.best_estimator_

In [49]:
val_analysis = val_ready.copy()

val_analysis["pred_tuned_gb"] = (
    best_model.predict(
        val_analysis[spo2_hr_features]
    )
)

In [50]:
val_analysis["abs_spo2_change_5min"] = (
    (
        val_analysis["future_spo2_5min"]
        - val_analysis["spo2"]
    )
    .abs()
)

In [51]:
val_analysis["change_group"] = pd.cut(
    val_analysis["abs_spo2_change_5min"],
    bins=[
        -np.inf,
        1,
        3,
        np.inf
    ],
    labels=[
        "Stable (<1 point)",
        "Moderate (1 to <3 points)",
        "Large (3+ points)"
    ],
    right=False
)

In [52]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

tuned_group_results = []

for group_name, group_df in val_analysis.groupby(
    "change_group",
    observed=True
):

    actual = group_df[target]

    predicted = group_df[
        "pred_tuned_gb"
    ]

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    tuned_group_results.append({
        "Change Group": group_name,
        "Rows": len(group_df),
        "MAE": mae,
        "RMSE": rmse
    })

tuned_group_results = pd.DataFrame(
    tuned_group_results
)

tuned_group_results

,Change Group,Rows,MAE,RMSE
0,Stable (<1 point),1559,0.266699,0.359527
1,Moderate (1 to <3 points),144,1.020587,1.331662
2,Large (3+ points),45,5.073127,7.699296


In [53]:
tuned_result = pd.DataFrame({
    "Model": [
        "Tuned Gradient Boosting - SpO2 + HR"
    ],
    "MAE": [
        tuned_mae
    ],
    "RMSE": [
        tuned_rmse
    ],
    "R2": [
        tuned_r2
    ]
})

tuned_result

,Model,MAE,RMSE,R2
0,Tuned Gradient Boosting - SpO2 + HR,0.45254,1.33695,0.151922


In [54]:
results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

model_results = pd.read_csv(
    results_file
)

model_results = model_results[
    model_results["Model"]
    != "Tuned Gradient Boosting - SpO2 + HR"
].copy()

model_results = pd.concat(
    [
        model_results,
        tuned_result
    ],
    ignore_index=True
)

model_results = (
    model_results
    .drop_duplicates(
        subset=["Model"],
        keep="last"
    )
    .reset_index(drop=True)
)

model_results.to_csv(
    results_file,
    index=False
)

model_results.sort_values("RMSE")

,Model,MAE,RMSE,R2
12,Tuned Gradient Boosting - SpO2 + HR,0.452540,1.336950,0.151922
9,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
8,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
11,XGBoost - SpO2 History,0.423909,1.362911,0.118666
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
4,Ridge - All Vitals,0.512447,1.384467,0.090568
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
7,Random Forest - All Vitals,0.472970,1.435870,0.021782


In [55]:
val_analysis = val_ready.copy()

val_analysis["pred_tuned_gb"] = best_model.predict(
    val_analysis[spo2_hr_features]
)

val_analysis["abs_spo2_change_5min"] = (
    val_analysis["future_spo2_5min"]
    - val_analysis["spo2"]
).abs()

val_analysis["change_group"] = pd.cut(
    val_analysis["abs_spo2_change_5min"],
    bins=[-np.inf, 1, 3, np.inf],
    labels=[
        "Stable (<1 point)",
        "Moderate (1 to <3 points)",
        "Large (3+ points)"
    ],
    right=False
)

In [56]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

tuned_group_results = []

for group_name, group_df in val_analysis.groupby(
    "change_group",
    observed=True
):

    actual = group_df[target]
    predicted = group_df["pred_tuned_gb"]

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    tuned_group_results.append({
        "Change Group": group_name,
        "Rows": len(group_df),
        "MAE": mae,
        "RMSE": rmse
    })

tuned_group_results = pd.DataFrame(
    tuned_group_results
)

tuned_group_results

,Change Group,Rows,MAE,RMSE
0,Stable (<1 point),1559,0.266699,0.359527
1,Moderate (1 to <3 points),144,1.020587,1.331662
2,Large (3+ points),45,5.073127,7.699296


In [57]:
tuned_group_file = (
    project_root
    / "data"
    / "processed"
    / "tuned_subgroup_results.csv"
)

tuned_group_results.to_csv(
    tuned_group_file,
    index=False
)

print("Tuned subgroup results saved!")

Tuned subgroup results saved!


In [58]:
import json

best_params = grid_search.best_params_

params_file = (
    project_root
    / "models"
    / "best_gradient_boosting_params.json"
)

with open(params_file, "w") as f:
    json.dump(
        best_params,
        f,
        indent=4
    )

print("Best parameters saved:")
print(best_params)
print(params_file)

Best parameters saved:
{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__min_samples_leaf': 3, 'model__n_estimators': 200}
c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\models\best_gradient_boosting_params.json
